In [1]:
!pip install duckdb
!pip install pandas
!pip install duckdb-engine jupysql
!pip install pathlib

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import duckdb
import pandas as pd 
from pathlib import Path


In [3]:
ROOT = Path().resolve().parent

con = duckdb.connect()
path_b = ROOT / "data" / "title.basics.tsv"
path_r = ROOT / "data" / "title.ratings.tsv"
path_w = ROOT / "data" / "watched.csv"
path_out = ROOT / "data" / "imdb_unseen.parquet"
path_out_user = ROOT / "data" / "user_profile.parquet"
path_demo = ROOT / "data" / "imdb_demo.parquet"

In [4]:
con.execute(f"""
    CREATE TABLE imdb AS
    SELECT 
        b.tconst,
        b.primaryTitle,
        b.originalTitle,
        b.startYear,
        b.runtimeMinutes,
        b.genres,
        r.averageRating,
        r.numVotes
    FROM read_csv('{path_b}', header=True, delim='\t', nullstr='\\N') b
    LEFT JOIN read_csv('{path_r}', header=True, delim='\t') r 
        ON b.tconst = r.tconst
    WHERE b.titleType = 'movie' AND b.isAdult = '0'
""")

In [5]:
con.execute("SELECT COUNT(*) FROM imdb").fetchone()

con.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(runtimeMinutes) AS com_duracao,
        COUNT(averageRating) as com_nota,
        COUNT(startYear) as com_ano
     FROM imdb
""").fetchdf()

,total,com_duracao,com_nota,com_ano
0,727550,459143,337864,617739


In [6]:
con.execute(f"""
    CREATE TABLE imdb_unseen AS
    SELECT i.*
    FROM imdb i
    LEFT JOIN read_csv('{path_w}', header=True, delim=',') w
        ON (
            LOWER(TRIM(i.primaryTitle)) = LOWER(TRIM(w.Name))
            OR
            LOWER(TRIM(i.originalTitle)) = LOWER(TRIM(w.Name))
            )
        AND TRY_CAST(i.startYear AS INTEGER) = w.Year
    WHERE w.Name IS NULL   
""")

In [7]:
con.execute("""
    SELECT       
        (SELECT COUNT(*) FROM imdb) AS total_imdb,
        (SELECT COUNT(*) FROM imdb_unseen) AS total_unseen
""").fetchdf()

,total_imdb,total_unseen
0,727550,727316


In [8]:
con.execute("""
    SELECT DISTINCT TRIM(genre) AS genre, COUNT(*) AS total
    FROM (
        SELECT UNNEST(STRING_SPLIT(genres,',')) AS genre
        FROM imdb_unseen
        WHERE genres IS NOT NULL
    )
    GROUP BY genre
    ORDER BY total DESC            
""").fetchdf()

,genre,total
0,Drama,268256
1,Documentary,145185
2,Comedy,121701
3,Action,60734
4,Romance,52680
5,Thriller,52145
6,Horror,45509
7,Crime,42307
8,Adventure,31391
9,Mystery,20057


In [9]:
con.execute("""
    ALTER TABLE imdb_unseen
    ALTER COLUMN startYear TYPE INTEGER USING TRY_CAST(startYear AS INTEGER);
""")
con.execute("""
    ALTER TABLE imdb_unseen
    ALTER COLUMN runtimeMinutes TYPE INTEGER USING TRY_CAST(runtimeMinutes AS INTEGER);
""")

In [10]:
con.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(runtimeMinutes) AS com_duracao,
        COUNT(averageRating) as com_nota,
        MIN(startYear) as ano_min,
        MAX(startYear) as ano_max,
        ROUND(AVG(averageRating),2) as media_nota
    FROM imdb_unseen
""").fetchdf()

,total,com_duracao,com_nota,ano_min,ano_max,media_nota
0,727316,458910,337631,1894,2031,6.16


In [13]:
con.execute(f"""
    CREATE TABLE watched_data AS
    SELECT 
        TRIM(w.Name) AS Name,
        w.Year,
        c.startYear,
        c.runtimeMinutes,
        c.genres
    FROM read_csv('{path_w}', header=True, delim=',') w
    LEFT JOIN imdb c ON (
            LOWER(TRIM(c.primaryTitle)) = LOWER(TRIM(w.Name))
            OR
            LOWER(TRIM(c.originalTitle)) = LOWER(TRIM(w.Name))
            )
        AND TRY_CAST(c.startYear AS INTEGER) = w.Year
""") 

In [17]:
con.execute("""
    CREATE OR REPLACE TABLE top_3_genres AS
    SELECT genre, count(*) as freq
    FROM (
        SELECT unnest(string_split(genres, ',')) as genre 
        FROM watched_data
    )
    GROUP BY 1 
    ORDER BY freq DESC 
    LIMIT 3;
""")

In [18]:
con.execute(f"""
    CREATE OR REPLACE TABLE user_metrics AS
    SELECT
        AVG(runtimeMinutes) AS avg_runtime,
        (SELECT list_aggregate(list(genre), 'string_agg', ', ') FROM top_3_genres) AS top_genres,
        (SELECT (CAST(startYear AS INTEGER) / 10) * 10 
            FROM watched_data 
            WHERE startYear IS NOT NULL
            GROUP BY 1 
            ORDER BY COUNT(*) DESC 
            LIMIT 1) AS top_decade
    FROM watched_data;
""")

con.execute(f"COPY user_metrics TO '{path_out_user}' (FORMAT PARQUET);")

In [ ]:
con.execute(f"""
    COPY (SELECT * FROM imdb_unseen) TO '{path_out}' (FORMAT 'parquet')
""")
print("Exportado com sucesso!")